# D⁺ → π⁺ π⁺ π⁻

```{autolink-concat}
```

The final state of $D^+\to\pi^+\pi^+\pi^-$ contains two **identical particles**, so the amplitude has to be symmetric under an exchange of the two $\pi^+$. This notebook checks that {mod}`ampform` and {mod}`ampform_dpd` formulate the same symmetric model, even though they impose the symmetry in different ways.

The equivalence check defines no dynamics; a Breit–Wigner model is only used for the [](#dalitz-plot) at the end.

In [ ]:
%matplotlib widget

In [ ]:
import logging
import os
import warnings

import ampform
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import qrules
import sympy as sp
from ampform.kinematics.lorentz import FourMomentumSymbol, InvariantMass
from IPython.display import Latex, Markdown
from matplotlib.colors import LogNorm
from tensorwaves.data.phasespace import TFPhaseSpaceGenerator
from tensorwaves.data.rng import TFUniformRealNumberGenerator
from tensorwaves.data.transform import SympyDataTransformer

from ampform_dpd import DalitzPlotDecompositionBuilder
from ampform_dpd.adapter.qrules import (
    normalize_state_ids,
    permute_equal_final_states,
    to_three_body_decay,
)
from ampform_dpd.dynamics.builder import BreitWignerBuilder
from ampform_dpd.io import (
    as_markdown_table,
    aslatex,
    cached,
    mute_ampform_warnings,
    simplify_latex_rendering,
)
from ampform_dpd.symmetrization import get_exchange_sign, get_exchange_ties

simplify_latex_rendering()
logging.getLogger("jax").setLevel(logging.ERROR)  # mute JAX
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # mute TF
warnings.simplefilter("ignore", category=RuntimeWarning)
if STATIC_PAGE := "EXECUTE_NB" in os.environ:
    mute_ampform_warnings()

## Decay definition

{mod}`qrules` removes permutations of equal final-state particles, so the permuted decay chains have to be generated explicitly with {func}`~ampform_dpd.adapter.qrules.permute_equal_final_states`. Both models are given the same, permuted set of transitions.

:::{note}
AmpForm performs this combinatorics itself, in {func}`~ampform.helicity.decay.perform_combinatorics`, so handing it the permuted transitions makes it count each permutation **twice** and its intensity comes out a factor $2^2$ too large. That factor is divided out in [](#confirm-equivalence). Passing the un-permuted reaction instead is not an option: AmpForm then leaves the kinematic variables of the permuted topology out of {attr}`~ampform.helicity.HelicityModel.kinematic_variables` and the model cannot be evaluated.
:::

In [ ]:
REACTION = qrules.generate_transitions(
    initial_state="D+",
    final_state=["pi+", "pi+", "pi-"],
    allowed_intermediate_particles=["rho(770)0", "f(0)(980)", "f(2)(1270)"],
    mass_conservation_factor=0,
    formalism="helicity",
)
REACTION = permute_equal_final_states(REACTION)
REACTION123 = normalize_state_ids(REACTION)

In [ ]:
src = qrules.io.asmermaid(REACTION123, collapse_graphs=True, markdown=True)
Markdown(src)

In [ ]:
DECAY = to_three_body_decay(REACTION123.transitions, min_ls=True)
Markdown(as_markdown_table([DECAY.initial_state, *DECAY.final_state.values()]))

In [ ]:
resonances = sorted(
    {t.resonance for t in DECAY.chains},
    key=lambda p: (p.name[0], p.mass),
)
resonance_names = [p.name for p in resonances]
Markdown(as_markdown_table(resonances))

Each resonance now occurs in **two** subsystems: with $\pi^+_1$ as spectator and with $\pi^+_2$ as spectator.

In [ ]:
Latex(aslatex(DECAY, with_jp=True))

## Model formulation

The two packages arrive at Bose symmetry in different ways:

- **AmpForm** sums the amplitude over both $\pi^+$ assignments and names its coefficients after the *particles* in the decay, not after their state IDs. The two permutations therefore share one coefficient and their sum is symmetric by construction.
- **AmpForm-DPD** writes every isobar in the cyclic pair ordering $(23)1, (31)2, (12)3$ of the DPD paper. Subsystem 1 then orders the isobar as $(\pi^+_2\pi^-)$, subsystem 2 as $(\pi^-\pi^+_1)$, so the exchange re-orders the isobar vertex and the couplings of the two subsystems have to be tied together with a relative sign. Computing that sign is what {mod}`~ampform_dpd.symmetrization` does.

AmpForm is thus an independent reference for the sign that AmpForm-DPD derives.

### DPD model

In [ ]:
model_builder = DalitzPlotDecompositionBuilder(DECAY, min_ls=True)
DPD_MODEL = model_builder.formulate(cleanup_summations=True)
DPD_MODEL.intensity.cleanup()

{meth}`~ampform_dpd.DalitzPlotDecompositionBuilder.formulate` imposes the symmetry by default, tying the couplings of the two subsystems of each resonance with {func}`~ampform_dpd.symmetrization.get_exchange_sign`. For this spinless final state that sign is $(-1)^{J_R}$:

In [ ]:
exchange_signs = {
    sp.Symbol(tie.chain.resonance.latex): get_exchange_sign(tie)
    for tie in get_exchange_ties(DECAY)
}
Latex(aslatex(exchange_signs))

:::{note} Where the sign comes from
The cyclic ordering writes the isobar of subsystem 1 as $(\pi^+_2\pi^-)$ and that of subsystem 2 as $(\pi^-\pi^+_1)$, so the exchange delivers the isobar's decay products in reversed order. Reversing them means $\theta \to \pi-\theta$, that is $\cos\theta_{31}(\sigma_2,\sigma_1) = -\cos\theta_{23}(\sigma_1,\sigma_2)$, and $P_J(-z) = (-1)^J P_J(z)$ turns that into a factor $(-1)^{J_R}$. In general the phase is $\eta = (-1)^{J_R-s_i-s_j}$, times the statistics sign $(-1)^{2s_a}$ of the exchanged particles -- see {mod}`~ampform_dpd.symmetrization`.
:::

Only the couplings of subsystem 1 survive as free parameters:

In [ ]:
Latex(aslatex(DPD_MODEL.parameter_defaults))

In [ ]:
Latex(aslatex(DPD_MODEL.amplitudes, terms_per_line=1))

### AmpForm model

Both $\pi^+$ assignments of a resonance therefore carry the same coefficient $C$:

In [ ]:
model_builder = ampform.get_builder(REACTION)
model_builder.config.use_helicity_couplings = False
model_builder.config.scalar_initial_state_mass = True
model_builder.config.stable_final_state_ids = {0, 1, 2}
AMPFORM_MODEL = model_builder.formulate()
AMPFORM_MODEL.intensity.cleanup()

In [ ]:
Latex(aslatex(AMPFORM_MODEL.amplitudes, terms_per_line=1))

## Phase space sample

Both models are evaluated over the same four-momentum sample. AmpForm's kinematic variables are defined in terms of the momenta $p_0,p_1,p_2$ directly, while the DPD variables have to be expressed in the Mandelstam variables first.

In [ ]:
p1, p2, p3 = (FourMomentumSymbol(f"p{i}", shape=[]) for i in (0, 1, 2))
s1, s2, s3 = sorted(DPD_MODEL.invariants, key=str)
mass_definitions = {
    **DPD_MODEL.masses,
    s1: InvariantMass(p2 + p3) ** 2,
    s2: InvariantMass(p1 + p3) ** 2,
    s3: InvariantMass(p1 + p2) ** 2,
}
dpd_variables = {
    symbol: expr.doit().xreplace(DPD_MODEL.variables).xreplace(mass_definitions)
    for symbol, expr in DPD_MODEL.variables.items()
}
dpd_variables.update({s: mass_definitions[s] for s in (s1, s2, s3)})
dpd_transformer = SympyDataTransformer.from_sympy(dpd_variables, backend="jax")
ampform_transformer = SympyDataTransformer.from_sympy(
    AMPFORM_MODEL.kinematic_variables, backend="jax"
)

In [ ]:
rng = TFUniformRealNumberGenerator(seed=0)
phsp_generator = TFPhaseSpaceGenerator(
    initial_state_mass=REACTION.initial_state[-1].mass,
    final_state_masses={i: p.mass for i, p in REACTION.final_state.items()},
)
PHSP = phsp_generator.generate(50_000, rng)
PHSP_EXCHANGED = {**PHSP, "p0": PHSP["p1"], "p1": PHSP["p0"]}

## Convert to numerical functions

Both models get the same couplings: AmpForm's coefficient $C$ of a resonance becomes that resonance's production coupling in the DPD model, with the DPD decay couplings set to 1. Each resonance gets a **different** complex value, so that the chains interfere -- the exchange signs are only visible in the interference.

In [ ]:
COUPLING_VALUES = {
    "rho(770)0": 1.0,
    "f(0)(980)": 0.8 + 0.6j,
    "f(2)(1270)": -0.5 + 0.9j,
}


def find_parameter(model, resonance, predicate):
    matches = [p for p in model.parameter_defaults if predicate(str(p))]
    match, *_ = [p for p in matches if resonance.latex in str(p)]
    return match


parameter_values = {}
for resonance in resonances:
    value = COUPLING_VALUES[resonance.name]
    coefficient = find_parameter(AMPFORM_MODEL, resonance, lambda s: s.startswith("C"))
    coupling = find_parameter(DPD_MODEL, resonance, lambda s: "production" in s)
    parameter_values[coefficient] = value
    parameter_values[coupling] = value
parameter_values.update({
    p: 1 for p in DPD_MODEL.parameter_defaults if "decay" in str(p)
})
Latex(aslatex(parameter_values))

In [ ]:
ampform_func = cached.lambdify(
    cached.unfold(AMPFORM_MODEL),
    parameters=AMPFORM_MODEL.parameter_defaults,
)
dpd_func = cached.lambdify(
    cached.unfold(DPD_MODEL),
    parameters=DPD_MODEL.parameter_defaults,
)
for func in (ampform_func, dpd_func):
    func.update_parameters({
        str(p): v for p, v in parameter_values.items() if str(p) in func.parameters
    })

In [ ]:
AMPFORM_PHSP = ampform_transformer(PHSP)
DPD_PHSP = dpd_transformer(PHSP)
AMPFORM_INTENSITIES = np.array(ampform_func(AMPFORM_PHSP)).real
DPD_INTENSITIES = np.array(dpd_func(DPD_PHSP)).real

## Bose symmetry

Exchanging the two $\pi^+$ means exchanging their momenta $p_0 \leftrightarrow p_1$. Both models are invariant under it:

In [ ]:
ampform_exchanged = np.array(ampform_func(ampform_transformer(PHSP_EXCHANGED))).real
dpd_exchanged = np.array(dpd_func(dpd_transformer(PHSP_EXCHANGED))).real
np.testing.assert_allclose(AMPFORM_INTENSITIES, ampform_exchanged, rtol=1e-12)
np.testing.assert_allclose(DPD_INTENSITIES, dpd_exchanged, rtol=1e-12)

With `symmetrize=False`, each subsystem keeps its own couplings. Giving those the same values -- the naive thing to do -- breaks the symmetry, because the $\rho(770)$ then enters the two subsystems with the wrong relative sign:

In [ ]:
asymmetric_builder = DalitzPlotDecompositionBuilder(DECAY, min_ls=True)
ASYMMETRIC_MODEL = asymmetric_builder.formulate(
    cleanup_summations=True, symmetrize=False
)
asymmetric_func = cached.lambdify(
    cached.unfold(ASYMMETRIC_MODEL),
    parameters=ASYMMETRIC_MODEL.parameter_defaults,
)
asymmetric_func.update_parameters({
    str(p): COUPLING_VALUES[r.name]
    for r in resonances
    for p in ASYMMETRIC_MODEL.parameter_defaults
    if "production" in str(p) and r.latex in str(p)
})
asymmetric = np.array(asymmetric_func(dpd_transformer(PHSP))).real
asymmetric_exchanged = np.array(asymmetric_func(dpd_transformer(PHSP_EXCHANGED))).real
largest_deviation = np.max(
    np.abs(asymmetric - asymmetric_exchanged) / np.abs(asymmetric)
)
assert largest_deviation > 1
Markdown(
    f"Without symmetrization, the intensity changes by up to"
    f" {100 * largest_deviation:.0f}% under the exchange of the two $\\pi^+$."
)

## Confirm equivalence

The two models agree over the whole phase space, once AmpForm's double counting (see [](#decay-definition)) is divided out:

In [ ]:
AMPFORM_DOUBLE_COUNTING = 2**2  # squared, because the intensity is |A|^2
np.testing.assert_allclose(
    DPD_INTENSITIES,
    AMPFORM_INTENSITIES / AMPFORM_DOUBLE_COUNTING,
    rtol=1e-10,
)

In [ ]:
ratio = DPD_INTENSITIES / AMPFORM_INTENSITIES
Markdown(
    f"The two intensities agree to a relative spread of"
    f" {np.std(ratio) / np.mean(ratio):.1e} over all {len(ratio):,} phase-space"
    f" points."
)

## Dalitz plot

Finally, a Breit–Wigner model to show what the symmetry looks like on the Dalitz plot. The intensity is evaluated on a regular $(\sigma_1,\sigma_2)$ grid, so that the projections below are exact instead of limited by phase-space statistics. Points outside the Dalitz-plot region evaluate to `nan` and are masked away.

In [ ]:
dynamics_builder = BreitWignerBuilder(
    energy_dependent_width=False,
    decay_form_factor=False,
    production_form_factor=False,
)
bw_builder = DalitzPlotDecompositionBuilder(DECAY, min_ls=True)
for chain in DECAY.chains:
    bw_builder.dynamics_choices.register_builder(chain, dynamics_builder)
BW_MODEL = bw_builder.formulate(cleanup_summations=True)
bw_func = cached.lambdify(
    cached.unfold(BW_MODEL), parameters=BW_MODEL.parameter_defaults
)
bw_func.update_parameters({
    str(p): COUPLING_VALUES[r.name]
    for r in resonances
    for p in BW_MODEL.parameter_defaults
    if "production" in str(p) and r.latex in str(p)
})

In [ ]:
angle_definitions = {
    symbol: expr.xreplace(BW_MODEL.masses)
    for symbol, expr in BW_MODEL.variables.items()
}
grid_transformer = SympyDataTransformer.from_sympy(angle_definitions, backend="jax")
MANDELSTAM_SUM = float(sum(m**2 for m in BW_MODEL.masses.values()))

X, Y = np.meshgrid(
    np.linspace(0, 3.1, num=500),
    np.linspace(0, 3.1, num=500),
)
grid_sample = {"sigma1": jnp.array(X), "sigma2": jnp.array(Y)}
grid_sample["sigma3"] = MANDELSTAM_SUM - grid_sample["sigma1"] - grid_sample["sigma2"]
grid_sample.update(grid_transformer(grid_sample))
Z = np.array(bw_func(grid_sample)).real

In [ ]:
sigma1 = R"$\sigma_1 = m^2(\pi^+_2\pi^-)$"
sigma2 = R"$\sigma_2 = m^2(\pi^+_1\pi^-)$"

fig, (ax_dalitz, ax_proj) = plt.subplots(
    figsize=(11, 4.5), ncols=2, layout="constrained"
)
fig.suptitle(R"$D^+ \to \pi^+\pi^+\pi^-$, Breit-Wigner model")

mesh = ax_dalitz.pcolormesh(X, Y, Z, norm=LogNorm(), rasterized=True)
ax_dalitz.plot([0, 3.1], [0, 3.1], c="white", ls="dotted", lw=1)
ax_dalitz.set_aspect("equal")
ax_dalitz.set_xlabel(f"{sigma1} [GeV$^2$]")
ax_dalitz.set_ylabel(f"{sigma2} [GeV$^2$]")
ax_dalitz.set_title("Dalitz-plot density")
fig.colorbar(mesh, ax=ax_dalitz)

density = np.nan_to_num(Z)
projections = {
    sigma1: (X[0], density.sum(axis=0), "red", "solid", 1.5),
    sigma2: (Y[:, 0], density.sum(axis=1), "blue", "dotted", 3.0),
}
for label, (x, y, color, style, width) in projections.items():
    ax_proj.plot(x, y, label=label, c=color, ls=style, lw=width)
ax_proj.set_xlabel(R"$\sigma$ [GeV$^2$]")
ax_proj.set_ylabel("Intensity [a.u.]")
ax_proj.set_title("Mass projections")
ax_proj.legend()
plt.show()

The Dalitz plot is mirror-symmetric about $\sigma_1=\sigma_2$ and the two projections lie exactly on top of each other, which is the statement of Bose symmetry for this final state:

In [ ]:
np.testing.assert_allclose(Z, Z.T, rtol=1e-10)
np.testing.assert_allclose(density.sum(axis=0), density.sum(axis=1), rtol=1e-10)